In [0]:
customers=spark.table("bronze.customers")
orders=spark.table("bronze.orders")
order_items=spark.table("bronze.order_items")
products=spark.table("bronze.products")
reviews=spark.table("bronze.reviews")
sellers=spark.table("bronze.sellers")
payments=spark.table("bronze.payments")


**CUSTOMERS TABLE CLEANING**


In [0]:
customers_silver=customers.dropDuplicates()
from pyspark.sql.functions import col

customers_silver.select([
    col(c).isNull().cast("int").alias(c)
    for c in customers_silver.columns
    ]).show()

customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.customers")

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
|          0|                 0|                       0|            0|             0|
|          0|                 0|                       0|            0|             0|
|          0|                 0|                       0|            0|             0|
|          0|                 0|                       0|            0|             0|
|          0|                 0|                       0|            0|             0|
|          0|                 0|                       0|            0|             0|
|          0|                 0|                       0|            0|             0|
|          0|                 0|           

**ORDERS TABLE CLEANING**

In [0]:
from pyspark.sql.functions import to_timestamp

orders_silver=orders\
    .withColumn("order_purchase_timestamp",to_timestamp("order_purchase_timestamp"))\
    .withColumn("order_delivered_customer_date",to_timestamp("order_delivered_customer_date"))

order_silver=orders_silver.dropDuplicates()

orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.orders")


**PAYMENTS TABLE CLEANING**

In [0]:
from pyspark.sql.functions import col

valid_payments=payments.filter(col("payment_value") > 0)
invalid_payments=payments.filter(col("payment_value") <= 0)


In [0]:
invalid_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.invalid_payments")

valid_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.payments")

**REVIEWS TABLE CLEANING**

In [0]:
valid_reviews=reviews.filter(col("review_score").between(1,5))
invalid_reviews=reviews.filter(~col("review_score").between(1,5))

valid_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.reviews")

**PRODUCTS TABLE CLEANING**

In [0]:
products_silver=products.dropDuplicates()

produccts_silver=products_silver.na.fill({"product_category_name":"Unknown"})

products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.products")


**SELLINGS TABLE CLEANING**

In [0]:
sellers_silver=sellers.dropDuplicates()

sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.sellers")


**ORDER ITEMS TABLE CLEANING**

In [0]:
order_items_silver=order_items.dropDuplicates()

order_items_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.order_items")

**DATA QUALITY METRICS**

In [0]:
from pyspark.sql import Row

dq=spark.createDataFrame([
    Row(metric="invalid_payments",count=invalid_payments.count()),
    Row(metric="invalid_reviews",count=invalid_reviews.count())
])

dq.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.data_quality_metrics")

In [0]:
spark.sql("SHOW TABLES IN silver").show(truncate=False)

+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|silver  |customers           |false      |
|silver  |data_quality_metrics|false      |
|silver  |invalid_payments    |false      |
|silver  |order_items         |false      |
|silver  |orders              |false      |
|silver  |payments            |false      |
|silver  |products            |false      |
|silver  |reviews             |false      |
|silver  |sellers             |false      |
+--------+--------------------+-----------+

